<a href="https://colab.research.google.com/github/DavidSenseman/BIO1173/blob/main/BIO1173_Class_02_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- BIO1173_CLASS_02_4 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **BIO 1173: Intro Computational Biology**

## **Module 2: Neural Networks with PyTorch**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)


#### Module 2 Material

* Class_02_1: Introduction to Neural Networks with PyTorch
* Class_02_2: Encoding Feature Vectors
* Class_02_3: Controlling Overfitting
* **Class_02_4: Saving and Loading a PyTorch Neural Network**

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which is your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    from google.colab import auth
    auth.authenticate_user()
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

If the code is correct, you should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "136.112.107.129",
  "hostname": "129.107.112.136.bc.googleusercontent.com",
  "city": "Council Bluffs",
  "region": "Iowa",
  "country": "US",
  "loc": "41.2619,-95.8608",
  "org": "AS396982 Google LLC",
  "postal": "51503",
  "timezone": "America/Chicago",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your Keys

Run the next cell to test whether your `myUTSA_ID` and your `BIO1173_Key` are correctly installed in your Colab Secrets. You will need to have both keys correctly installed in your Colab Secrets in order to submit your work for grading using Electronic Submission.

In [ ]:
# @title Test Your Keys

from google.colab import userdata
import os

# Check if myUTSA_ID is properly loaded
try:
    # 1. Get the key from Secrets
    myUTSA_ID = userdata.get('myUTSA_ID')

    # 2. Set it as an environment variable
    os.environ['myUTSA_ID'] = myUTSA_ID

    # print("myUTSA_ID is loaded and environment variable set successfully!")
    print(f"myUTSA_ID: {myUTSA_ID}")

except Exception as e:
    print(f"Error loading myUTSA_ID: {e}")
    print("Please set your myUTSA_ID in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'myUTSA_ID'")
    print("3. Paste your myUTSA_ID and toggle 'Notebook access' on")

# Check if BIO1173 key is properly loaded
try:
    # 1. Get the key from Secrets
    BIO1173_KEY = userdata.get('BIO1173_KEY')

    # 2. Set it as an environment variable
    os.environ['BIO1173_KEY'] = BIO1173_KEY

    #print("BIO1173 key loaded and environment variable set successfully!")
    print(f"BIO1173_KEY: {BIO1173_KEY}")

except Exception as e:
    print(f"Error loading BIO1173 key: {e}")
    print("Please set your BIO1173 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'BIO1173_KEY'")
    print("3. Paste your BIO1173 key and toggle 'Notebook access' on")


If you have correctly installed your myUTSA id in your Colab Secrets, you should see something _similar_ to following but with your information:
```text
myUTSA_ID: abc123
BIO1173_KEY: BIO1173-F26-04-999-ABC-DEF
```
However, if you see an error message, you will need to fix this problem before you can submit your lesson for grading. Ask your Instructor or TA for help if you can't resolve the problem yourself --- that's what they are here for, to help you with course problems.

## Create Custom Function

Run the cell below to create a function needed for this lesson.

In [ ]:
# @title Create Custom Function
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)

# **Saving and Loading a PyTorch Neural Network**

Complex neural networks will take a _long_ time to train.  It is helpful to be able to save a trained neural network so that you can reload it and using it again--a reloaded neural network will **not** require retraining.  

PyTorch provides the following formats for saving neural networks:

* **State Dict** - Stores the model's state dictionary (weights and biases) in a format that can be easily loaded back into the same model architecture.
* **Full Model** - Stores the complete neural network including both the model architecture and weights in a single file.

Usually, you will want to save using the state dict approach as it's more flexible and memory efficient. The state dict only saves the parameters (weights and biases) while keeping the model class definition separate, allowing you to load the weights into different model architectures if needed.

A primary goal of this lesson is to convince you that a PyTorch model that you "regenerate" from a file that was saved to your Google Drive, is exactly the same (i.e. same accuracy) as the original PyTorch model you created and trained.

## Example 1A: Build and Train a Classification Neural Network

The code in `Example 1A` builds, compiles and trains a neural network called `model_eg` that can classify the Quality of an orange based on its physical and chemical characteristics.

The code in the cell below reads the Orange Quality dataset from the course HTTP server and creates a DataFrame called `df_eg` (i.e. "orange" DataFrame) using this code snippet:

```Python
df_eg = pd.read_csv(
"https://biologicslab.co/BIO1173/data/orange_quality.csv",
    na_values=['NA', '?'])
```

In order to create a feature vector, the 3 non-numeric columns in the dataset: `Color`, `Variety` and `Blemished` must be pre-processed as follows:
1. **Mapping strings to integers:** This is used to take care of the column `Color` which contains several string values. Here is the code snippet that does the mapping:

```Python
# Map str to int
mapping_eg = {'Orange':0,'Deep Orange':1,'Light Orange':2,'Orange-Red':3,'Yellow-Orange':4} df_eg['Color'] = df_eg['Color'].map(mapping_eg)
```

2. **Normalization:** The following code chunk identifies which columns in `df_eg` are numeric and then applies Z-normalization to the numeric values.
    
```Python
# Standardise all numeric column with z‑score
numeric_cols_eg = df_eg.select_dtypes(include=['int64', 'float64']).columns df_eg[numeric_cols_eg] = df_eg[numeric_cols_eg].apply(zscore) numeric_cols = df_eg.select_dtypes(include=['int64', 'float64']).columns
```

3. **Exclude Columns:** To take care the string values in the columns `Variety` and `Blemished`, we will simply exclude them from the list of columns to be used for creating the X feature vector `X_eg`:

```Python
# Generate X-values
X_eg = df_eg[['Size (cm)', 'Weight (g)', 'Brix (Sweetness)', 'pH (Acidity)','Softness (1-5)', 'HarvestTime (days)', 'Ripeness (1-5)','Color']].values X_eg = np.asarray(X_eg).astype('float32')
```

Since we are building a classification neural network, we will need to one-hot encode the column `Quality (1-5)` which contains the `Y-values` using this code snippet:

```Python
# Generate y-values (one-hot encoding)
dummies_eg = pd.get_dummies(df_eg['Quality (1-5)'], dtype=int)
y_eg = dummies_eg.values
y_eg = np.asarray(y_eg).astype('float32')
```

It should be noted that this column is already numeric, so we are **not** using one-hot encoding to replace string values with integer. Rather, one-hot encoding the `Y-values` is necessary to give them the **correct format** for the neural network.


In [ ]:
# @title Example 1A: Build and Train a Classification Neural Network

# Imports
# ------------------------------------------------------------
import pandas as pd
import time
import numpy as np
from scipy.stats import zscore
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Parameters
# ------------------------------------------------------------
EPOCHS        = 100
PATIENCE      = 10
VERBOSE       = 2

# Load data
# ------------------------------------------------------------
df_eg = pd.read_csv(
    "https://biologicslab.co/BIO1173/data/orange_quality.csv",
    na_values=['NA', '?'])

#  Preprocessing
# ------------------------------------------------------------

# Map str to int
mapping_eg = {'Orange':0,'Deep Orange':1,
           'Light Orange':2,'Orange-Red':3,
           'Yellow-Orange':4}
df_eg['Color'] = df_eg['Color'].map(mapping_eg)

# Standardise all numeric column with z‑score
numeric_cols_eg = df_eg.select_dtypes(include=['int64', 'float64']).columns
df_eg[numeric_cols_eg] = df_eg[numeric_cols_eg].apply(zscore)


# Generate X-values
X_eg = df_eg[['Size (cm)', 'Weight (g)', 'Brix (Sweetness)', 'pH (Acidity)',
       'Softness (1-5)', 'HarvestTime (days)', 'Ripeness (1-5)',
        'Color']].values
X_eg = np.asarray(X_eg).astype('float32')

# Generate y-values
dummies_eg = pd.get_dummies(df_eg['Quality (1-5)'], dtype=int) # Classification
y_eg = dummies_eg.values
y_eg = np.asarray(y_eg).astype('float32')

# Prepare data for PyTorch
# ------------------------------------------------------------

# Convert to PyTorch tensors
X_tensor_eg = torch.FloatTensor(X_eg)
y_tensor_eg = torch.LongTensor(np.argmax(y_eg, axis=1))  # Convert one-hot to class indices

# Split into train and validation sets
X_train_eg, X_val_eg, y_eg_train, y_eg_val = train_test_split(
    X_tensor_eg, y_tensor_eg, test_size=0.2, random_state=42
)

# Create data loaders
train_dataset_eg = TensorDataset(X_train_eg, y_eg_train)
val_dataset_eg = TensorDataset(X_val_eg, y_eg_val)
train_loader_eg = DataLoader(train_dataset_eg, batch_size=32, shuffle=True)
val_loader_eg = DataLoader(val_dataset_eg, batch_size=32, shuffle=False)


# Define model
# ------------------------------------------------------------

class OrangeQualityModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(OrangeQualityModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)  # No softmax here - it's included in CrossEntropyLoss
        return x

# Create model instance
input_dim_eg = X_train_eg.shape[1]
num_classes_eg = y_eg.shape[1]
model_eg = OrangeQualityModel(input_dim_eg, num_classes_eg)

# Define optimizer and loss function
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_eg.parameters(), lr=0.001)

# Training loop with early stopping and history tracking
# ------------------------------------------------------------

# Initialize history tracking lists
train_losses_eg = []
val_losses_eg = []
train_acc_accuracies = []
val_acc_accuracies = []

# Early stopping variables
best_val_acc = 0.0
patience_counter = 0

print(f"------Training Starting for {EPOCHS} epochs --------------")
start_time = time.time()

for epoch in range(EPOCHS):
    # Training phase
    model_eg.train()
    train_loss_eg = 0.0
    train_correct_eg = 0
    train_total_eg = 0

    for inputs, labels in train_loader_eg:
        optimizer.zero_grad()
        outputs = model_eg(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss_eg += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total_eg += labels.size(0)
        train_correct_eg += (predicted == labels).sum().item()

    # Validation phase
    model_eg.eval()
    val_loss_eg = 0.0
    val_correct_eg = 0
    val_total_eg = 0

    with torch.no_grad():
        for inputs, labels in val_loader_eg:
            outputs = model_eg(inputs)
            loss = criterion(outputs, labels)

            val_loss_eg += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total_eg += labels.size(0)
            val_correct_eg += (predicted == labels).sum().item()

    # Calculate accuracies
    train_acc_eg = 100. * train_correct_eg / train_total_eg
    val_acc_eg = 100. * val_correct_eg / val_total_eg

    # Store history for plotting
    train_losses_eg.append(train_loss_eg/len(train_loader_eg))
    val_losses_eg.append(val_loss_eg/len(val_loader_eg))
    train_acc_accuracies.append(train_acc_eg)
    val_acc_accuracies.append(val_acc_eg)

    if VERBOSE > 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}]')
        print(f'Train Loss: {train_loss_eg/len(train_loader_eg):.4f}, '
              f'Train Acc: {train_acc_eg:.2f}%')
        print(f'Val Loss: {val_loss_eg/len(val_loader_eg):.4f}, '
              f'Val Acc: {val_acc_eg:.2f}%')
        print('-' * 50)

    # Early stopping logic
    if val_acc_eg > best_val_acc:
        best_val_acc = val_acc_eg
        patience_counter = 0
        # Save best model
        torch.save(model_eg.state_dict(), "best_classification_model_eg.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Inspect training
# ---------------------------------------------------------------------------

print(f"\nTraining finished.")
print(f"Best validation accuracy: {best_val_acc:.4f}")

elapsed_time = time.time() - start_time
print("Elapsed time: {}".format(hms_string(elapsed_time)))

# Show the best validation accuracy and loss (same as original)
best_val_acc_history = max(val_acc_accuracies) if val_acc_accuracies else 0.0
best_val_loss_history = min(val_losses_eg) if val_losses_eg else float('inf')

print(f"Best validation accuracy: {best_val_acc_history:.4f}")
print(f"Best validation loss: {best_val_loss_history:.4f}")

# Load the best model for final evaluation if needed
model_eg.load_state_dict(torch.load("best_classification_model_eg.pth"))

If the code is correct you should see something _similar_ to the following output:
```text
------Training Starting for 100 epochs --------------
Epoch [1/100]
Train Loss: 2.1741, Train Acc: 11.98%
Val Loss: 2.0051, Val Acc: 36.73%
--------------------------------------------------
Epoch [2/100]
Train Loss: 1.9803, Train Acc: 23.44%
Val Loss: 1.9141, Val Acc: 38.78%
--------------------------------------------------
Epoch [3/100]
Train Loss: 1.8705, Train Acc: 32.81%
Val Loss: 1.8018, Val Acc: 42.86%
--------------------------------------------------
Epoch [4/100]
Train Loss: 1.7007, Train Acc: 41.15%
Val Loss: 1.7037, Val Acc: 40.82%
--------------------------------------------------
Epoch [5/100]
Train Loss: 1.6324, Train Acc: 43.23%
Val Loss: 1.6343, Val Acc: 40.82%
--------------------------------------------------
Epoch [6/100]
Train Loss: 1.5967, Train Acc: 47.92%
Val Loss: 1.5880, Val Acc: 40.82%
--------------------------------------------------
Epoch [7/100]
Train Loss: 1.5159, Train Acc: 45.83%
Val Loss: 1.5581, Val Acc: 38.78%
--------------------------------------------------
Epoch [8/100]
Train Loss: 1.4745, Train Acc: 53.65%
Val Loss: 1.5308, Val Acc: 38.78%
--------------------------------------------------
Epoch [9/100]
Train Loss: 1.4001, Train Acc: 53.65%
Val Loss: 1.5119, Val Acc: 38.78%
--------------------------------------------------
Epoch [10/100]
Train Loss: 1.4067, Train Acc: 53.65%
Val Loss: 1.4939, Val Acc: 42.86%
--------------------------------------------------
Epoch [11/100]
Train Loss: 1.3682, Train Acc: 55.21%
Val Loss: 1.4704, Val Acc: 38.78%
--------------------------------------------------
Epoch [12/100]
Train Loss: 1.3735, Train Acc: 54.17%
Val Loss: 1.4564, Val Acc: 40.82%
--------------------------------------------------
Epoch [13/100]
Train Loss: 1.3067, Train Acc: 56.25%
Val Loss: 1.4405, Val Acc: 42.86%
--------------------------------------------------
Early stopping at epoch 13

Training finished.
Best validation accuracy: 42.8571
Elapsed time: 0:00:00.26
Best validation accuracy: 42.8571
Best validation loss: 1.4405
<All keys matched successfully>
```

The `model_eg` neural network trained very quickly (< 1 min) but the best validation accuracy (`val accuracy`) is only about 40-45%. It should also be noted that in this particular run, `Early Stopping` terminated training before the 20th epoch.

## Example 1B: Plot Training History

The code in the cell below generates two side-by-side plots, an **Accuracy Curve** and a **Loss Curve**. These curves provide a visual way to follow what happened during training of `model_eg`.

In [ ]:
# @title Example 1B: Plot Training History

import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))

# Main title above both subplots
plt.suptitle('Training Visualization: Orange Quality Dataset',
             fontsize=16, fontweight='medium')

plt.subplot(1,2,1)
plt.plot(train_acc_accuracies, label='train')
plt.plot(val_acc_accuracies, label='val')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_losses_eg, label='train')
plt.plot(val_losses_eg, label='val')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

If the code is correct you should see something _similar_ to the following output:

![__](https://biologicslab.co:/BIO1173/images/class_02/class_02_4_image01A.png)


## **Exercise 1A: Build and Train a Classification Neural Network**

In the cell below build and train a new classification neural network called `model_ex`. The goal of your neural network model (`model_ex`) will be to classify apples in the Apple Quality dataset using the values in the following columns: 'Size', 'Weight', 'Sweetness', 'Crunchiness', 'Juiciness', 'Acidity' and 'Ripeness'.

**Code Hints:**

1. Copy-and-paste Example 1 into the code cell below.
2. Change the code that reads the data set to read as shown in the following code chunk:
```Python
df_ex = pd.read_csv(
    "https://biologicslab.co/BIO1173/data/apple_quality.csv",
    na_values=['NA', '?'])

```
3. Remove the Preprocessing code. Since all of these columns are numeric, there is no need to use the preprocessing code in Example 1. In this situation, it's better to comment-out the unecessary code than to actually cut it out. Commenting-out code sections has the distinct advantage that it is easy to correct any mistakes.

While you could add a 'hastag' (**#**) before each line of code, an easier way is to use a group three single quotation marks (**' ' '**) instead. Just place a group immediately _before_  the code you what to exclude, and place another group immediately _after_  the code as shown in this example:
```Python
#  Preprocessing
# ------------------------------------------------------------

'''
# Map str to int
mapping_eg = {'Orange':0,'Deep Orange':1,
           'Light Orange':2,'Orange-Red':3,
           'Yellow-Orange':4}
df_eg['Color'] = df_ex['Color'].map(mapping_eg)
'''
```
In Colab, the code that is commented out is displayed in red.

4. Change the values for `X_ex` and `y_ex` to match the apple quality data in `df_ex`. Use the code in the following code chunk to define your X and y values.
```Python
# Generate X-values
X_ex = df_ex[['Size', 'Weight', 'Sweetness', 'Crunchiness',
       'Juiciness', 'Acidity', 'Ripeness']].values
X_ex = np.asarray(X_ex).astype('float32')

# Generate y-values
dummies_ex = pd.get_dummies(df_ex['Quality'], dtype=int) # Classification
y_ex = dummies_ex.values
y_ex = np.asarray(y_ex).astype('float32')
```
4. Change the code that contructs your `model_ex` neural network. Use this code chunk to create your model for **Exercise 1A**:
```Python
# Define model
# ------------------------------------------------------------

class AppleQualityModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(AppleQualityModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)  # No softmax here - it's included in CrossEntropyLoss
        return x

# Create model instance
input_dim_ex = X_train_ex.shape[1]
num_classes_ex = y_ex.shape[1]
model_ex = AppleQualityModel(input_dim_ex, num_classes_ex)
```

5. Finally, make sure that you change every instance of the suffix `_eg` to `_ex`.

In [ ]:
# @title Exercise 1A: Build and Train a Classification Neural Network




If the code is correct you should see something _similar_ to the following output:
```text
------Training Starting for 100 epochs --------------
Epoch [1/100]
Train Loss: 0.4865, Train Acc: 75.72%
Val Loss: 0.3543, Val Acc: 86.25%
--------------------------------------------------
Epoch [2/100]
Train Loss: 0.3867, Train Acc: 82.56%
Val Loss: 0.3081, Val Acc: 88.62%
--------------------------------------------------
Epoch [3/100]
Train Loss: 0.3563, Train Acc: 83.50%
Val Loss: 0.2972, Val Acc: 89.12%
--------------------------------------------------
Epoch [4/100]
Train Loss: 0.3474, Train Acc: 84.34%
Val Loss: 0.2827, Val Acc: 89.38%
--------------------------------------------------
Epoch [5/100]
Train Loss: 0.3366, Train Acc: 85.25%
Val Loss: 0.2776, Val Acc: 89.38%
--------------------------------------------------
Epoch [6/100]
Train Loss: 0.3315, Train Acc: 84.72%
Val Loss: 0.2798, Val Acc: 88.50%
--------------------------------------------------
Epoch [7/100]
Train Loss: 0.3305, Train Acc: 85.41%
Val Loss: 0.2737, Val Acc: 89.50%
--------------------------------------------------
Epoch [8/100]
Train Loss: 0.3247, Train Acc: 85.56%
Val Loss: 0.2755, Val Acc: 89.12%
--------------------------------------------------
Epoch [9/100]
Train Loss: 0.3239, Train Acc: 85.47%
Val Loss: 0.2561, Val Acc: 90.25%
--------------------------------------------------
Epoch [10/100]
Train Loss: 0.3111, Train Acc: 86.66%
Val Loss: 0.2602, Val Acc: 90.75%
--------------------------------------------------
Epoch [11/100]
Train Loss: 0.3110, Train Acc: 86.28%
Val Loss: 0.2532, Val Acc: 90.75%
--------------------------------------------------
Epoch [12/100]
Train Loss: 0.3196, Train Acc: 85.97%
Val Loss: 0.2536, Val Acc: 91.00%
--------------------------------------------------
Epoch [13/100]
Train Loss: 0.3065, Train Acc: 86.75%
Val Loss: 0.2475, Val Acc: 91.00%
--------------------------------------------------
Epoch [14/100]
Train Loss: 0.3084, Train Acc: 86.19%
Val Loss: 0.2482, Val Acc: 91.38%
--------------------------------------------------
Epoch [15/100]
Train Loss: 0.2948, Train Acc: 87.62%
Val Loss: 0.2437, Val Acc: 91.75%
--------------------------------------------------
Epoch [16/100]
Train Loss: 0.3025, Train Acc: 87.09%
Val Loss: 0.2435, Val Acc: 91.00%
--------------------------------------------------
Epoch [17/100]
Train Loss: 0.2940, Train Acc: 86.69%
Val Loss: 0.2374, Val Acc: 92.75%
--------------------------------------------------
Epoch [18/100]
Train Loss: 0.2864, Train Acc: 87.47%
Val Loss: 0.2328, Val Acc: 92.38%
--------------------------------------------------
Epoch [19/100]
Train Loss: 0.2916, Train Acc: 87.88%
Val Loss: 0.2281, Val Acc: 92.62%
--------------------------------------------------
Epoch [20/100]
Train Loss: 0.2763, Train Acc: 87.94%
Val Loss: 0.2273, Val Acc: 92.62%
--------------------------------------------------
Epoch [21/100]
Train Loss: 0.2760, Train Acc: 88.25%
Val Loss: 0.2231, Val Acc: 92.75%
--------------------------------------------------
Epoch [22/100]
Train Loss: 0.2676, Train Acc: 87.91%
Val Loss: 0.2289, Val Acc: 93.12%
--------------------------------------------------
Epoch [23/100]
Train Loss: 0.2755, Train Acc: 87.72%
Val Loss: 0.2235, Val Acc: 93.00%
--------------------------------------------------
Epoch [24/100]
Train Loss: 0.2798, Train Acc: 88.25%
Val Loss: 0.2191, Val Acc: 93.50%
--------------------------------------------------
Epoch [25/100]
Train Loss: 0.2840, Train Acc: 88.44%
Val Loss: 0.2148, Val Acc: 92.88%
--------------------------------------------------
Epoch [26/100]
Train Loss: 0.2731, Train Acc: 88.78%
Val Loss: 0.2188, Val Acc: 93.00%
--------------------------------------------------
Epoch [27/100]
Train Loss: 0.2691, Train Acc: 88.56%
Val Loss: 0.2083, Val Acc: 92.88%
--------------------------------------------------
Epoch [28/100]
Train Loss: 0.2804, Train Acc: 88.50%
Val Loss: 0.2124, Val Acc: 93.75%
--------------------------------------------------
Epoch [29/100]
Train Loss: 0.2620, Train Acc: 89.44%
Val Loss: 0.2095, Val Acc: 93.25%
--------------------------------------------------
Epoch [30/100]
Train Loss: 0.2618, Train Acc: 88.62%
Val Loss: 0.2087, Val Acc: 93.50%
--------------------------------------------------
Epoch [31/100]
Train Loss: 0.2439, Train Acc: 89.91%
Val Loss: 0.2007, Val Acc: 93.00%
--------------------------------------------------
Epoch [32/100]
Train Loss: 0.2570, Train Acc: 89.22%
Val Loss: 0.2074, Val Acc: 93.50%
--------------------------------------------------
Epoch [33/100]
Train Loss: 0.2548, Train Acc: 89.12%
Val Loss: 0.1997, Val Acc: 92.88%
--------------------------------------------------
Epoch [34/100]
Train Loss: 0.2463, Train Acc: 89.97%
Val Loss: 0.2029, Val Acc: 93.25%
--------------------------------------------------
Epoch [35/100]
Train Loss: 0.2589, Train Acc: 89.50%
Val Loss: 0.1979, Val Acc: 93.00%
--------------------------------------------------
Epoch [36/100]
Train Loss: 0.2407, Train Acc: 89.56%
Val Loss: 0.1970, Val Acc: 93.88%
--------------------------------------------------
Epoch [37/100]
Train Loss: 0.2511, Train Acc: 89.53%
Val Loss: 0.2006, Val Acc: 93.88%
--------------------------------------------------
Epoch [38/100]
Train Loss: 0.2279, Train Acc: 90.56%
Val Loss: 0.1969, Val Acc: 93.88%
--------------------------------------------------
Epoch [39/100]
Train Loss: 0.2444, Train Acc: 89.72%
Val Loss: 0.1980, Val Acc: 93.12%
--------------------------------------------------
Epoch [40/100]
Train Loss: 0.2474, Train Acc: 89.41%
Val Loss: 0.1981, Val Acc: 93.38%
--------------------------------------------------
Epoch [41/100]
Train Loss: 0.2502, Train Acc: 89.62%
Val Loss: 0.1992, Val Acc: 93.00%
--------------------------------------------------
Epoch [42/100]
Train Loss: 0.2351, Train Acc: 90.62%
Val Loss: 0.1926, Val Acc: 93.62%
--------------------------------------------------
Epoch [43/100]
Train Loss: 0.2518, Train Acc: 89.25%
Val Loss: 0.1980, Val Acc: 92.62%
--------------------------------------------------
Epoch [44/100]
Train Loss: 0.2398, Train Acc: 89.75%
Val Loss: 0.1934, Val Acc: 93.88%
--------------------------------------------------
Epoch [45/100]
Train Loss: 0.2306, Train Acc: 90.53%
Val Loss: 0.1943, Val Acc: 93.50%
--------------------------------------------------
Epoch [46/100]
Train Loss: 0.2353, Train Acc: 90.09%
Val Loss: 0.1933, Val Acc: 93.50%
--------------------------------------------------
Early stopping at epoch 46

Training finished.
Best validation accuracy: 93.8750
Elapsed time: 0:00:11.64
Best validation accuracy: 93.8750
Best validation loss: 0.1926
<All keys matched successfully>

```

Your `model_ex` neural network appears to have done a much better job with a best validation accuracy (`val accuracy`) over 90%. It should also be noted that in this particular run, `Early Stopping` terminated training at the $46th Epoch.

## **Exercise 1B: Visualize Training**

In the cell below write the code to generate two side-by-side plots, an **Accuracy Curve** and a **Loss Curve** for your `model_ex` using Example 1B as a model.

**Code Hints:**

1. Copy-and-paste Example 1B into the cell below.
2. Change the main title to read as follows:
```Python
# Main title above both subplots
plt.suptitle('Training Visualization: Apple Quality Dataset',
             fontsize=16, fontweight='medium')

```

In [ ]:
# @title Exercise 1B: Visualize Training




If the code is correct you should see something _similar_ to the following output:

![__](https://biologicslab.co:/BIO1173/images/class_02/class_02_4_image02A.png)


## **Time and Cost of Training Large Language Models (LLMs)**

Large Language Models (LLMs) require enormous time and money to train. Below is the best publicly available data as of July 2026. Note: most labs no longer disclose parameter counts, GPU-hours, or exact training costs for their flagship models — this secrecy is itself a recent trend worth discussing with students.

#### **Notable LLM Training Runs**

| Model | Approx. size (parameters) | Rough training duration / compute | Rough training cost (estimates) | Source |
|-------|---------------------------|-------------------------------------|----------------------------------|--------|
| **Claude Fable 5** (Anthropic, June 2026) | Undisclosed | Undisclosed | Undisclosed | Anthropic has not published these figures for any Claude model |
| **GPT-5.6 Sol** (OpenAI, July 2026) | Undisclosed | Undisclosed | Undisclosed (API pricing: $5/$30 per 1M input/output tokens) | OpenAI has not published model size or training cost |
| **Llama 3.1 405B** (Meta, 2024) | **405 B** (confirmed) | ~54 days pretraining, 39.3M cumulative GPU-hours, 16,000+ H100 GPUs, 15T tokens | **~$170 M** (estimate) | Meta model card; Epoch AI |
| **DeepSeek-V3** (DeepSeek, 2024) | **671 B total / 37 B active** (MoE, confirmed) | 2.79 M GPU-hours on H800 GPUs | **~$5.6 M** (compute only — excludes R&D staff and experimentation) | DeepSeek technical report |
| **GPT-4** (OpenAI, 2023, for historical contrast) | ~1.8 T total (MoE) — *leaked, unconfirmed* | ~90–100 days on ~25,000 A100 GPUs | **$40 M – $100 M+** | Epoch AI; Stanford AI Index 2025 |

#### **Quick take-aways**

1. **The newest frontier models are a black box.** For Claude Fable 5 and GPT-5.6 Sol — the two most recent frontier releases as of this lesson — neither Anthropic nor OpenAI has disclosed parameter counts, training duration, GPU counts, or training cost. All that's public is API pricing (what it costs *you* to use the model), not what it cost the company to *build* it.

2. **This is a change from a few years ago.** Meta (Llama) still publishes real numbers — 405B parameters, 39.3 million GPU-hours, confirmed. So does DeepSeek. But since roughly GPT-4 (2023), the very largest labs (OpenAI, Google, Anthropic) have treated model size and training cost as trade secrets. Good discussion question for students: *why might a company disclose this for one model and not another?* (Competitive advantage, safety concerns, and export-control considerations are all part of the real answer.)

3. **Indirect signals still exist.** Even without official numbers, you can infer scale from things like API pricing tiers, context window size, and benchmark compute modes (e.g., Sol's "ultra" mode spinning up parallel subagents) — a good way to teach students to reason from indirect evidence.

4. **Where hard numbers exist, they're staggering.** Llama 3.1 405B alone used enough cumulative GPU-hours to equal roughly 4,500 GPU-years — only feasible because of massive parallelization across thousands of GPUs simultaneously.

> **Bottom line:**
> A few years ago, students could look up exact parameter counts and rough training costs for the top LLMs. Today, for the newest frontier models like Claude Fable 5 and GPT-5.6 Sol, that information simply isn't public — a shift worth flagging as part of the lesson itself, not just a gap in the data.

## **Determine the Model's RMSE and Accuracy**

The overall objective of this assignment is to convince you that you can save a trained neural network to a file, and then later, recreate the neural network from the file, without changing the model's accuracy.

**Why is this important?**

As you already know, it can take significant time and processing power to train even relatively small neural networks that we created so far in this course. Neural networks that are used commercially (think "Siri" or "Alexa" or ChatGPT) are many times larger and require enormous resources as well as weeks (or months) to train. Obviously, if you had to train a neural network every time you wanted to use it, it won't be very practical and there would be little interest in "AI". However, once the neural network has been trained, you can save it to a file, and then re-use it over and over again, without any loss in the neural network's ability to solve problems (i.e. loss in accuracy).

The code in the cell below calculates the ability of the `model_eg` neural network to predict an orange's quality (y-value) based on its physical and chemical characteristics (X-values). Two measures of predictive ability are computed, the Root Mean Square Error (RMSE) and Accuracy. The code stores the RMSE value in the variable `Score_eg` and the Accuracy value in the variable `Correct_eg` and then prints out these values.


## Example 2: Determine the Model's RMSE and Accuracy

The code in the cell below calculates 4 **accuracy metrics** about the `model_eg` before we save it to disk.

In [ ]:
# @title Example 2: Determine the model's accuracy before saving to disk

import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score

# Print title
print("Measurements of Orange NN model `model_eg`")
print("--------------------------------------------")

# Measure RMSE error (using cross-entropy loss as proxy since we're doing classification)
# For classification, we typically don't use RMSE directly, but here's how you'd calculate it:
with torch.no_grad():
    pred_eg = model_eg(X_tensor_eg).numpy()  # Get predictions from model
    score_eg = np.sqrt(metrics.mean_squared_error(pred_eg, y_eg))
    print(f"Before save score (RMSE): {score_eg}")

# Measure the accuracy
predict_classes_eg = np.argmax(pred_eg,axis=1)
expected_classes_eg = np.argmax(y_eg,axis=1)
correct_eg = accuracy_score(expected_classes_eg,predict_classes_eg)
print(f"Before save Accuracy: {correct_eg}")

If the code is correct you should see something _similar_ to the following output:
```text
Measurements of Orange NN model `model_eg`
--------------------------------------------
Before save score (RMSE): 0.7893741105679465
Before save Accuracy: 0.5601659751037344
```

We are going to use these scores to see if the model we save--and then redeploy--has the same accuracy or does the model's accuracy suffer?

### **Exercise 2: Determine the Model's Accuracy**

In the cell below, write the code to calculate the same **accuracy metrics** for your `ap_model` shown in `Example 2` before we save it to disk.

**Code HintsL**

1. Copy-and-paste Example 2 into the cell below.
2. Change the title to read as follows:
```Python

# Print title
print("Measurements of Apple NN model `ap_model`")
print("--------------------------------------------")
```

3. Change every instance of the suffix `_eg` to `_ex`.

In [ ]:
# @title Exercise 2: Determine the Model's Accuracy



If your code is correct you should see something _similar_ to the following output:
```text
Measurements of Apple NN model `ap_model`
--------------------------------------------
Before save score (RMSE): 1.8567140917425444
Before save Accuracy: 0.9385
```
According to the output shown above, your `model_ex` is better than 90% accurate when it comes to predicting apple quality. Apparently, it's a little easier to predict an apple's `Quality` with a classification neural network than to predict the quality of an orange.

# **Storing a PyTorch model to a Disk Drive**

There are two basic ways to save a PyTorch model to a disk drive, the primary difference lies in **what is stored** and **how you load it**.

### **PyTorch state_dict**
* **What it is:** A standard Python dictionary that maps each layer name to its specific parameter tensors (weights and biases) and buffers (like BatchNorm running means).
* **Storage:** It stores only the parameters, not the model's architecture or source code.
* **Loading:** To use it, you must first initialize the model in your code (e.g., model = MyModel()) and then call model.load_state_dict(torch.load(PATH)).
Best For: Production and sharing models. It is the recommended approach because it is flexible and doesn't break if you move your files around or refactor your project.

### **Full Model (torch.save(model, PATH))**
* **What it is:** A serialized version of the entire model object using Python’s pickle module.
* **Storage:** It saves the architecture, parameters, and the file path to the class definition.
* **Loading:** You can load it directly with model = torch.load(PATH) without defining the class first in your script.

### Example 3: Save the Model

The code in the cell below saves the trained neural network `model_eg` as a file in two different file formats: **`PyTorch state dict`** and **`full model`**.

Each file is saved in the current working directory (save_path = "."). The filename of the state dict file is `dict_model_eg_pytorch.pth` while the filename of the full model file is `full_model_eg_pytorch.pth`.

In [ ]:
# @title Example 3: Save the model

import os
import torch

# Save path is the current directory
save_path = "."

# Save the PyTorch model state dict (weights and architecture)
torch.save(model_eg.state_dict(), os.path.join(save_path, "dict_model_eg_pytorch.pth"))

# Save the entire model (architecture + weights)
torch.save(model_eg, os.path.join(save_path, "full_model_eg_pytorch.pth"))

# Print out the files in current directory
files = os.listdir()
print(files)


If your code is correct you should see something _similar_ to the following output:

```text
['.config', 'dict_model_eg_pytorch.pth', 'full_model_eg_pytorch.pth', 'drive', 'sample_dat
```
After running the code cell above, there should now be two new files in your current directory, `'dict_model_eg_pytorch.pth'` and `full_model_eg_pytorch.pth`.

## **Exercise 3: Save the Model**

In the code cell below save your _trained_ neural network `model_ex` as a dictionary (`dict_model_ex_pytorch.pth`) and as a full model (`full_model_eg_pytorch.pth`).

In [ ]:
# @title Exercise 3: Save the Model



You should now see the two more files with your neural network, `dict_model_ex_pytorch.pth` and `full_model_ex_pytorch.pth`.

**WARNING:** Do not continue unless you can see **4 files:** 2 files for the example model (`model_eg`) and 2 files for the exercise model (`model_ex)`.

## Example 4: Create New Model from Saved Model

Once a trained model has been saved, it is a simple matter to read the file to make an exact copy of the model. In PyTorch, we need to recreate the model architecture first and then load the saved weights into it.

In Example 4, we have recreated the neural network architecture and loaded the saved weights into a new model instance called `model_new_eg` to differentiate it from the one that was built previously, `model_eg`.

In [ ]:
# @title Example 4: Create new model from saved model

import torch
from torch import nn
import os
FILENAME = "dict_model_eg_pytorch.pth"

print("=== Model Loading Verification ===")

# Print details of model
print(f"Training data shapes:")
print(f"X shape: {X_eg.shape}")
print(f"Y shape: {y_eg.shape}")

# Check dimensions
num_classes = y_eg.shape[1]
input_dim = X_eg.shape[1]

print(f"Input features: {input_dim}")
print(f"Number of classes: {num_classes}")

# Recreate the EXACT same model class structure as used in training
class OrangeQualityModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(OrangeQualityModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Create model with EXACT same dimensions as training
model_new_eg = OrangeQualityModel(input_dim, num_classes)  # input_dim=8, num_classes=8

try:
    print(f"Loading weights from saved model: '{FILENAME}'")
    model_new_eg.load_state_dict(torch.load(FILENAME))
    print("✓ Orange Quality model loaded successfully!")

except Exception as e:
    print(f"✗ Error loading model: {e}")

If your code is correct you should see the following output:
```text
=== Model Loading Verification ===
Training data shapes:
X shape: (241, 8)
Y shape: (241, 8)
Input features: 8
Number of classes: 8
Loading weights from saved model: 'dict_model_eg_pytorch.pth'
✓ Orange Quality model loaded successfully!
```

## **Exercise 4: Create New Model from Saved Model**

In the cell below create a new neural network called `ap2_model` from the file `ap_model_pytorch.pth` in your current directory. Print out a summary of your new `ap2_model`.

**Code Hints:**

1. Copy-and-paste Example 4 into the cell below.
2. Change the filename as follows:
```Python
FILENAME = "dict_model_ex_pytorch.pth"
```
3. Change the model structure to read as shownn in the following code chunk:
```Python
# Recreate the EXACT same model class structure as used in training
class AppleQualityModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(AppleQualityModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x
```

4. Change the model as shown in this code snippet:
```Python
# Create model with EXACT same dimensions as training
model_new_ex = AppleQualityModel(input_dim, num_classes)
```
5. Change every instance of the suffix `_eg` to `_ex`.

In [ ]:
# @title Exercise 4: Create New Model from Saved Model



If your code is correct you should see the following output:

```text
=== Model Loading Verification ===
Training data shapes:
X shape: (4000, 7)
Y shape: (4000, 2)
Input features: 7
Number of classes: 2
Loading weights from saved model: 'dict_model_ex_pytorch.pth'
✓ Apple Quality model loaded successfully!
```

## Example 5: Compare the Predictive Accuracy of the Original and the New Model

The code in the cell below computes the RMSE error and the Accuracy of our new model `model_new_eg` and compares these values with the original `model_eg`.

We are trying to address the question whether re-loaded model has the same accuracy as the original model?

In [ ]:
# Example 5: Compare the Predictive Accuracy of the Old and New Models

# Set both models to evaluation mode for consistent predictions
model_eg.eval()  # Make sure original is in eval mode
model_new_eg.eval()  # Make sure new model is also in eval mode

print("\n=== MODEL VERIFICATION ===")
with torch.no_grad():
    # Get predictions from both models (both should be in eval mode now)
    pred_eg = model_eg(X_tensor_eg).numpy()
    pred_new_eg = model_new_eg(X_tensor_eg).numpy()

    print(f"Original model prediction shape: {pred_eg.shape}")
    print(f"New model prediction shape: {pred_new_eg.shape}")

    # Show first few predictions for comparison
    orig_preds = np.argmax(pred_eg, axis=1)
    new_preds = np.argmax(pred_new_eg, axis=1)

    print(f"\nOriginal model (first 5): {orig_preds[:5]}")
    print(f"New model      (first 5): {new_preds[:5]}")

    # Check exact match
    are_equal = np.array_equal(orig_preds, new_preds)
    print(f"\nPredictions identical: {are_equal}")

    if not are_equal:
        diff_count = np.sum(orig_preds != new_preds)
        print(f"Number of different predictions: {diff_count}")

        # Show some examples where they differ
        diff_indices = np.where(orig_preds != new_preds)[0]
        if len(diff_indices) > 0:
            print("First few differing indices:", diff_indices[:5])
            for i in range(min(5, len(diff_indices))):
                idx = diff_indices[i]
                print(f"Index {idx}: Original={orig_preds[idx]}, New={new_preds[idx]}")

print("\n=== Model Architecture Check ===")
print("Original model:")
print(model_eg)
print("\nNew loaded model:")
print(model_new_eg)

# Let's also check the weights are identical
print("\n=== Weight Comparison ===")
weight_diffs = []
for (name1, param1), (name2, param2) in zip(model_eg.named_parameters(), model_new_eg.named_parameters()):
    diff = torch.sum(torch.abs(param1 - param2)).item()
    weight_diffs.append(diff)
    print(f"{name1}: difference = {diff:.2e}")

total_weight_diff = sum(weight_diffs)
print(f"\nTotal weight difference: {total_weight_diff:.2e}")


If the code is correct the output you should see something _similar_ to the following output:`

```text

=== MODEL VERIFICATION ===
Original model prediction shape: (241, 8)
New model prediction shape: (241, 8)

Original model (first 5): [7 6 7 3 7]
New model      (first 5): [7 6 7 3 7]

Predictions identical: True

=== Model Architecture Check ===
Original model:
OrangeQualityModel(
  (fc1): Linear(in_features=8, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=8, bias=True)
)

New loaded model:
OrangeQualityModel(
  (fc1): Linear(in_features=8, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=8, bias=True)
)

=== Weight Comparison ===
fc1.weight: difference = 0.00e+00
fc1.bias: difference = 0.00e+00
bn1.weight: difference = 0.00e+00
bn1.bias: difference = 0.00e+00
fc2.weight: difference = 0.00e+00
fc2.bias: difference = 0.00e+00
bn2.weight: difference = 0.00e+00
bn2.bias: difference = 0.00e+00
fc3.weight: difference = 0.00e+00
fc3.bias: difference = 0.00e+00
```

As you can see, there is **_no difference_** in the accuracy of the saved model compared to the original one.

>> ### **Train Once...**
>> ### **Use Anywhere!**

Big generative models like `ChatGTP` can take days months to train in massive data centers. But once trained trained and saved, they can process new data (**inference**) very fast (>$100$ tokens/second).

## **Exercise 5: Compare the Predictive Accuracy of the Old and New Models**

In the cell below write the code to compute the RMSE and Accuracy values for your `model_new_ex` and print out these values along with the values for your original `model_ex`.

**Code Hints:**

1. Copy-and-paste Example 5 into the cell below.
2. Change every instance of the suffix `_eg` with `_ex`.

In [ ]:
# Exercise 5: Compare the Predictive Accuracy of the Old and New Models



If the code is correct, you should see something _similar_ to the following output:
```text

=== MODEL VERIFICATION ===
Original model prediction shape: (4000, 2)
New model prediction shape: (4000, 2)

Original model (first 5): [1 1 0 1 1]
New model      (first 5): [1 1 0 1 1]

Predictions identical: True

=== Model Architecture Check ===
Original model:
AppleQualityModel(
  (fc1): Linear(in_features=7, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
)

New loaded model:
AppleQualityModel(
  (fc1): Linear(in_features=7, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
)

=== Weight Comparison ===
fc1.weight: difference = 0.00e+00
fc1.bias: difference = 0.00e+00
bn1.weight: difference = 0.00e+00
bn1.bias: difference = 0.00e+00
fc2.weight: difference = 0.00e+00
fc2.bias: difference = 0.00e+00
bn2.weight: difference = 0.00e+00
bn2.bias: difference = 0.00e+00
fc3.weight: difference = 0.00e+00
fc3.bias: difference = 0.00e+00

Total weight difference: 0.00e+00
```

## **Summary: Pre-trained Model Persistence and Its Importance for Large Language Models**

The ability to save and reload trained PyTorch neural networks without any loss of accuracy represents a fundamental breakthrough in machine learning workflow efficiency. When we train a complex neural network, especially one with millions or billions of parameters like modern large language models (LLMs), the training process can take weeks or months using powerful computational resources. However, once training is complete, we can save the model's weights and architecture to disk, creating a portable file that captures all the learned knowledge.

This capability becomes critically important in the era of extremely large LLMs because these models require enormous computational resources for training - often costing thousands of dollars in cloud computing time and consuming massive amounts of energy. By saving trained models, practitioners can:
1. Avoid retraining identical or similar architectures from scratch
2. Deploy models to production environments immediately
3. Share pre-trained models with others in the research community
4. Fine-tune existing models on new datasets without starting from random weights

For LLMs specifically, this persistence allows researchers and developers to build upon existing knowledge rather than reinventing the wheel each time. A model trained on general text can be saved, then fine-tuned for specific applications like medical diagnosis, legal document analysis, or creative writing - all while maintaining the original model's accuracy and performance characteristics. This approach democratizes access to sophisticated AI capabilities by making powerful pre-trained models accessible to organizations that cannot afford expensive training infrastructure, ultimately accelerating innovation in artificial intelligence.


# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/BIO1173/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()